# 🧬 MedAIChain — ML Prediction Server (Kaggle Edition)

Ce notebook lance le service Flask de prédiction (rendez-vous lab) **directement sur Kaggle**
et l'expose via **ngrok** pour que votre backend NestJS puisse l'appeler.

### Routes disponibles
| Méthode | Route | Description |
|---------|-------|-------------|
| `GET`   | `/health` | État du service + modèles chargés |
| `POST`  | `/predict` | Modèle « tier » (`model.pkl`) |
| `POST`  | `/predict-ml-api` | 2ᵉ modèle (`model_ml_api.pkl`) |

### Étapes
1. **Uploadez** `model.pkl` (et optionnellement `model_ml_api.pkl`) comme *Dataset* Kaggle.
2. **Renseignez** votre token ngrok dans la cellule de configuration.
3. **Exécutez toutes les cellules** — l'URL publique s'affichera.
4. **Copiez** cette URL dans la config de votre backend NestJS.

---
## ⚙️ 1 — Installation des dépendances

In [ ]:
!pip install flask pyngrok joblib pandas scikit-learn --quiet

---
## 🔑 2 — Configuration

1. Créez un compte gratuit sur [ngrok.com](https://ngrok.com)
2. Copiez votre **Authtoken** depuis le dashboard ngrok
3. Collez-le ci-dessous

In [ ]:
# ============================================================
# 🔧  CONFIGURATION — modifiez ces valeurs
# ============================================================

NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTH_TOKEN_HERE"   # ← collez votre token ngrok ici
NGROK_DOMAIN = ""   # ← (Optionnel) collez votre domaine statique ngrok gratuit ici\n
# Chemins vers les fichiers modèle (.pkl)
# Sur Kaggle, un dataset uploadé se trouve dans /kaggle/input/<dataset-name>/
# Adaptez les chemins ci-dessous selon le nom de votre dataset.
MODEL_PKL_PATH       = "/kaggle/input/medaichain-models/model.pkl"
MODEL_ML_API_PKL_PATH = "/kaggle/input/medaichain-models/model_ml_api.pkl"

FLASK_PORT = 5000

---
## 📦 3 — Chargement des modèles

In [ ]:
import os
import joblib

# --- Modèle principal (obligatoire) ---
if not os.path.isfile(MODEL_PKL_PATH):
    raise FileNotFoundError(
        f"❌ Fichier modèle introuvable : {MODEL_PKL_PATH}\n"
        "   → Uploadez model.pkl comme Dataset Kaggle et vérifiez le chemin."
    )

model = joblib.load(MODEL_PKL_PATH)
print(f"✅ Modèle principal chargé : {MODEL_PKL_PATH}")

# --- 2ᵉ modèle (optionnel) ---
model_ml_api = None
if os.path.isfile(MODEL_ML_API_PKL_PATH):
    model_ml_api = joblib.load(MODEL_ML_API_PKL_PATH)
    print(f"✅ Modèle ml-api chargé  : {MODEL_ML_API_PKL_PATH}")
else:
    print(f"⚠️  model_ml_api.pkl non trouvé — la route /predict-ml-api sera désactivée.")

---
## 🚀 4 — Serveur Flask + Tunnel ngrok

Cette cellule démarre le serveur Flask dans un **thread** et ouvre un tunnel ngrok.
L'URL publique s'affichera en sortie — utilisez-la dans votre backend NestJS.

In [ ]:
import threading
import pandas as pd
from flask import Flask, request, jsonify
from pyngrok import ngrok

# ──────────────────────────────────────────────
# Urgency keywords
# ──────────────────────────────────────────────
URGENCY_KEYWORDS = (
    "urgent", "urgence", "urgences", "critique",
    "immédiat", "immediate", "asap",
    "prioritaire", "priorité", "priorite",
    "grave", "douleur intense",
)

def _note_indique_urgence(note_lower: str) -> bool:
    return any(m in note_lower for m in URGENCY_KEYWORDS)

# ──────────────────────────────────────────────
# Flask App
# ──────────────────────────────────────────────
app = Flask(__name__)


@app.route("/health", methods=["GET"])
def health():
    return jsonify({
        "status": "ok",
        "service": "ml_predict_kaggle",
        "models": {
            "tier": os.path.basename(MODEL_PKL_PATH),
            "ml_api": os.path.basename(MODEL_ML_API_PKL_PATH) if model_ml_api is not None else None,
        },
    })


@app.route("/predict", methods=["POST"])
def predict():
    data = request.json or {}

    note              = str(data.get("note", "")).lower().strip()
    type_analyse      = data.get("type_analyse", "")
    allergies         = str(data.get("allergies", "")).strip()
    subscription_tier = str(data.get("subscription_tier", "free")).lower().strip()

    nb_allergies = 0 if allergies == "" else len(allergies.split("|"))

    tier_map = {"free": 0, "plus": 1, "premium": 2}
    tier_score = tier_map.get(subscription_tier, 0)

    df = pd.DataFrame([{
        "note":          note,
        "type_analyse":  type_analyse,
        "nb_allergies":  nb_allergies,
    }])

    pred   = model.predict(df)[0]
    result = "Acceptée automatiquement" if pred == 1 else "En attente"

    return jsonify({
        "result":            result,
        "subscription_tier": subscription_tier,
        "tier_score":        tier_score,
    })


@app.route("/predict-ml-api", methods=["POST"])
def predict_ml_api():
    if model_ml_api is None:
        return jsonify({
            "error": "model_ml_api.pkl introuvable",
            "hint":  "Uploadez model_ml_api.pkl dans votre Dataset Kaggle",
        }), 503

    try:
        data = request.get_json()
        if data is None:
            return jsonify({"error": "No JSON received"}), 400

        note         = data.get("note", "")
        type_analyse = data.get("type_analyse", "")
        allergies    = data.get("allergies", "")

        if type_analyse == "":
            return jsonify({"error": "type_analyse is required"}), 400

        note      = str(note).lower().strip()
        allergies = str(allergies).strip()

        if note == "":
            return jsonify({"result": "⏳ En attente"})

        # Urgence explicite → acceptée
        if _note_indique_urgence(note):
            return jsonify({
                "result":     "Acceptée automatiquement",
                "prediction": 1,
                "reason":     "urgence_note",
            })

        nb_allergies = 0
        if allergies != "":
            nb_allergies = len(allergies.split("|"))

        df = pd.DataFrame([{
            "note":          note,
            "type_analyse":  type_analyse,
            "nb_allergies":  nb_allergies,
        }])

        pred   = model_ml_api.predict(df)[0]
        result = "Acceptée automatiquement" if pred == 1 else "⏳ En attente"

        return jsonify({"result": result, "prediction": int(pred)})

    except Exception as e:
        return jsonify({"error": "Internal server error", "details": str(e)}), 500


# ──────────────────────────────────────────────
# Start Flask in a background thread
# ──────────────────────────────────────────────
def run_flask():
    app.run(host="0.0.0.0", port=FLASK_PORT, use_reloader=False)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()
print(f"🟢 Flask démarré sur le port {FLASK_PORT}")

# ──────────────────────────────────────────────
# Open ngrok tunnel
# ──────────────────────────────────────────────
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
if NGROK_DOMAIN:\n    public_url = ngrok.connect(FLASK_PORT, domain=NGROK_DOMAIN)\nelse:\n    public_url = ngrok.connect(FLASK_PORT)\n
print("="*60)
print(f"🌐 URL PUBLIQUE ngrok :  {public_url}")
print("="*60)
print()
print("Utilisez cette URL dans votre backend NestJS, par exemple :")
print(f"  ML_SERVICE_URL={public_url}")
print()
print("Routes :")
print(f"  GET  {public_url}/health")
print(f"  POST {public_url}/predict")
print(f"  POST {public_url}/predict-ml-api")
print()
print("⚠️  Gardez ce notebook en cours d'exécution pour que le tunnel reste actif !")

---
## 🧪 5 — Test rapide (optionnel)

Exécutez cette cellule pour vérifier que tout fonctionne.

In [ ]:
import requests, json

base = f"http://localhost:{FLASK_PORT}"

# --- Health check ---
r = requests.get(f"{base}/health")
print("── /health ──")
print(json.dumps(r.json(), indent=2, ensure_ascii=False))

# --- /predict ---
payload_predict = {
    "note": "bilan sanguin complet",
    "type_analyse": "sang",
    "allergies": "pénicilline|aspirine",
    "subscription_tier": "premium",
}
r = requests.post(f"{base}/predict", json=payload_predict)
print("\n── /predict ──")
print(json.dumps(r.json(), indent=2, ensure_ascii=False))

# --- /predict-ml-api ---
payload_ml = {
    "note": "douleur intense au thorax",
    "type_analyse": "radio",
    "allergies": "",
}
r = requests.post(f"{base}/predict-ml-api", json=payload_ml)
print("\n── /predict-ml-api ──")
print(json.dumps(r.json(), indent=2, ensure_ascii=False))

---
## 🛑 6 — Arrêter le tunnel

Exécutez cette cellule lorsque vous avez terminé pour fermer proprement le tunnel ngrok.

In [ ]:
ngrok.kill()
print("🔴 Tunnel ngrok fermé.")